# Tiers, escalation ladder and Markov statistics (replayable)

Every image is a Merkle tree from pixel rows up to the frame; every step decision is a custody record; the Markov statistics are recomputed from the chain. Swap `CHAIN` and `IMAGES` for your own data.

In [ ]:
import json, numpy as np, cv2
from pathlib import Path
from baebugs import tiers as T, ladder as L
from baebugs.mmr import MMRLog
DEMO = Path('/Volumes/magicBLACKbox/baebugs/ladder/demo')   # or docs/ladder_demo in the repo
if not DEMO.exists(): DEMO = Path('../docs/ladder_demo')
CHAIN = DEMO / 'ladder_mmr.jsonl'

## 1. One image, every tier
Pixel rows → 16 px tiles → 2×2 groups → whole image; objects carry their bbox and the tiles they cover.

In [ ]:
rng = np.random.default_rng(0)
img = rng.integers(0, 255, (240, 320, 3), dtype=np.uint8)
fr = T.frame(img, [{'id': 'spec1', 'bbox': [100, 60, 180, 140], 'label': 'Coleoptera'}], meta={'step': 'image'})
print(json.dumps(T.summary(fr)['tiers']), fr['root'][:16])

## 2. Break points: compare two snapshots top-down
Only unequal subtrees are opened; the first unequal tiles are the break points.

In [ ]:
img2 = img.copy(); img2[90:110, 120:150] = 0
d = T.diff(fr, T.frame(img2, [{'id': 'spec1', 'bbox': [100, 60, 180, 140], 'label': 'Coleoptera'}], meta={'step': 'image'}))
print(d['break_boxes'], d['objects'], f"{d['nodes_compared']} of {d['nodes_total']} nodes compared")

## 3. Prove a single pixel against the frame

In [ ]:
p = T.prove_pixel(img, fr, 130, 95)
print('pixel', p['pixel'], 'verifies:', T.verify_pixel(p))
p['pixel'] = [0, 0, 0]; print('tampered verifies:', T.verify_pixel(p))

## 4. The ladder chain from the toy demo
Verify the MMR, then recompute the Markov statistics and compare with the stored hash.

In [ ]:
log = MMRLog(CHAIN)
print('chain verifies:', log.verify(), 'records:', len(log.records))
stats = L.markov(log.records)
stored = json.loads((DEMO / 'markov.json').read_text())
print('stats recomputed == stored:', stats['stats_sha256'] == stored['stats_sha256'])
print(json.dumps(stats['decided_by'], indent=1))

In [ ]:
import pandas as pd
pd.DataFrame(stats['transitions'])

In [ ]:
pd.DataFrame(stats['approval_by_step'])

## 5. Snapshots

In [ ]:
import matplotlib.pyplot as plt
snaps = sorted((DEMO / 'snapshots').glob('*.png'))[:9]
fig, ax = plt.subplots(3, 3, figsize=(14, 10))
for a, p in zip(ax.flat, snaps):
    a.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)); a.set_title(p.name); a.axis('off')
plt.tight_layout()